In [78]:

from __future__ import annotations
import os
from dataclasses import dataclass
from typing import Literal
import numpy as np
import pandas as pd



@dataclass
class BuildConfig:

    train_events_path: str = "../data/processed/chicago_clean_2015_2024.parquet"
    test_events_path: str = "../data/processed/chicago_clean_2025.parquet"


    out_train_table: str = "../data/processed/train_table_train_2015_2024.parquet"
    out_test_table: str = "../data/processed/train_table_test_2025.parquet"

 
    date_col: str = "date"
    lat_col: str = "latitude"
    lon_col: str = "longitude"


    time_freq: Literal["W", "D"] = "W"

    task: Literal["classification", "regression"] = "classification"


In [79]:

# Module 1) load_events: read + validate + parse date
def load_events(path: str, cfg: BuildConfig) -> pd.DataFrame:
    df = pd.read_parquet(path)

    required = [cfg.date_col, cfg.lat_col, cfg.lon_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns {missing} in {path}. "
            f"Available columns: {df.columns.tolist()}"
        )

    df = df.copy()
    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col], errors="coerce")
    df = df.dropna(subset=[cfg.date_col]).copy()

    return df

In [80]:

# Module 2) make_time_bin: create time window index 
def make_time_bin(df: pd.DataFrame, date_col: str, freq: str) -> pd.DataFrame:
    df = df.copy()

    if freq == "D":
        df["time_bin"] = df[date_col].dt.floor("D")
    elif freq == "W":

        df["time_bin"] = df[date_col].dt.to_period("W").dt.start_time
    else:
        raise ValueError(f"Unsupported freq: {freq}. Use 'W' or 'D'.")

    return df

In [81]:

# Module 3) aggregate_panel: CITY-LEVEL aggregation (baseline)

def aggregate_panel_city(df: pd.DataFrame) -> pd.DataFrame:
    panel = (
        df.groupby(["time_bin"], as_index=False)
          .size()
          .rename(columns={"size": "crime_count"})
    )
    panel["crime_count"] = panel["crime_count"].astype(int)
    panel["cell_id"] = "city"
    panel = panel[["cell_id", "time_bin", "crime_count"]]
    return panel

In [82]:

# Module 4) complete_panel: fill missing time windows 
def complete_panel(panel: pd.DataFrame, freq: str) -> pd.DataFrame:
    panel = panel.copy()

    cells = panel["cell_id"].unique()
    tmin, tmax = panel["time_bin"].min(), panel["time_bin"].max()

    if freq == "D":
        full_time = pd.date_range(tmin, tmax, freq="D")
    elif freq == "W":
        full_time = pd.date_range(tmin, tmax, freq="7D")
    else:
        raise ValueError(f"Unsupported freq: {freq}. Use 'W' or 'D'.")

    full_index = pd.MultiIndex.from_product([cells, full_time], names=["cell_id", "time_bin"])

    out = (
        panel.set_index(["cell_id", "time_bin"])
             .reindex(full_index)
             .fillna(0)
             .reset_index()
    )
    out["crime_count"] = out["crime_count"].astype(int)
    return out

In [83]:

# Module 5) make_label: future window label (anti-leakage)
def make_label(panel: pd.DataFrame, task: str) -> pd.DataFrame:
    out = panel.sort_values(["cell_id", "time_bin"]).copy()
    g = out.groupby("cell_id", group_keys=False)

    out["y_next_count"] = g["crime_count"].shift(-1)

    out["hist_mean"] = (
        g["crime_count"]
        .shift(1)                    
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
    )

    out = out.dropna(subset=["y_next_count", "hist_mean"]).copy()

    if task == "classification":
        out["y"] = (out["y_next_count"] > out["hist_mean"]).astype(int)
    else:
        out["y"] = out["y_next_count"].astype(int)

    return out

In [84]:
# Module 6) make_features: lag + rolling + calendar features (anti-leakage)

def make_features(panel_with_y: pd.DataFrame) -> pd.DataFrame:
    out = panel_with_y.sort_values(["cell_id", "time_bin"]).copy()
    g = out.groupby("cell_id", group_keys=False)

    dt = pd.to_datetime(out["time_bin"])
    out["year"] = dt.dt.year
    out["month"] = dt.dt.month
    out["weekofyear"] = dt.dt.isocalendar().week.astype(int)
    out["dayofweek"] = dt.dt.dayofweek

    out["lag_1"] = g["crime_count"].shift(1)
    out["lag_2"] = g["crime_count"].shift(2)
    out["lag_4"] = g["crime_count"].shift(4)

    base = g["crime_count"].shift(1)
    out["roll_mean_4"] = base.rolling(4).mean()
    out["roll_sum_4"] = base.rolling(4).sum()
    out["roll_mean_8"] = base.rolling(8).mean()
    out["roll_sum_8"] = base.rolling(8).sum()
    out["roll_std_8"] = base.rolling(8).std()

    feat_cols = [
        "lag_1", "lag_2", "lag_4",
        "roll_mean_4", "roll_sum_4",
        "roll_mean_8", "roll_sum_8", "roll_std_8"
    ]
    out[feat_cols] = out[feat_cols].fillna(0.0)

    return out

In [85]:

# finalize: keep trainable columns only

def finalize_table(df: pd.DataFrame) -> pd.DataFrame:
    keep_cols = [
        "cell_id", "time_bin", "y",
        "year", "month", "weekofyear", "dayofweek",
        "lag_1", "lag_2", "lag_4",
        "roll_mean_4", "roll_sum_4",
        "roll_mean_8", "roll_sum_8", "roll_std_8"
    ]
    out = df[keep_cols].copy()

    feature_cols = [c for c in out.columns if c not in ["cell_id", "time_bin", "y"]]
    for c in feature_cols:
        out[c] = (
            pd.to_numeric(out[c], errors="coerce")
              .replace([np.inf, -np.inf], np.nan)
              .fillna(0.0)
        )

    out["y"] = pd.to_numeric(out["y"], errors="coerce").fillna(0).astype(int)
    return out


def save_parquet(df: pd.DataFrame, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)



# pipeline

def build_one(events_path: str, cfg: BuildConfig) -> pd.DataFrame:
    events = load_events(events_path, cfg)
    events = make_time_bin(events, cfg.date_col, cfg.time_freq)

    panel = aggregate_panel_city(events)


    panel = complete_panel(panel, cfg.time_freq)

    panel = make_label(panel, cfg.task)
    panel = make_features(panel)
    table = finalize_table(panel)
    return table


def main():
    cfg = BuildConfig()

    train_table = build_one(cfg.train_events_path, cfg)
    test_table = build_one(cfg.test_events_path, cfg)

    save_parquet(train_table, cfg.out_train_table)
    save_parquet(test_table, cfg.out_test_table)

    print("Train table:", train_table.shape, " y mean:", float(train_table["y"].mean()))
    print("Test table :", test_table.shape, " y mean:", float(test_table["y"].mean()))
    print(train_table.head())


if __name__ == "__main__":
    main()

Train table: (521, 15)  y mean: 0.5335892514395394
Test table : (51, 15)  y mean: 0.7058823529411765
  cell_id   time_bin  y  year  month  weekofyear  dayofweek   lag_1   lag_2  \
1    city 2015-01-05  1  2015      1           2          0     0.0     0.0   
2    city 2015-01-12  1  2015      1           3          0   835.0     0.0   
3    city 2015-01-19  1  2015      1           4          0  1062.0   835.0   
4    city 2015-01-26  0  2015      1           5          0  1123.0  1062.0   
5    city 2015-02-02  0  2015      2           6          0   946.0  1123.0   

   lag_4  roll_mean_4  roll_sum_4  roll_mean_8  roll_sum_8  roll_std_8  
1    0.0          0.0         0.0          0.0         0.0         0.0  
2    0.0          0.0         0.0          0.0         0.0         0.0  
3    0.0          0.0         0.0          0.0         0.0         0.0  
4    0.0          0.0         0.0          0.0         0.0         0.0  
5  835.0        991.5      3966.0          0.0         0.0 